<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 32 · Automated Reporting and Communication

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch32_automated_reporting_communication.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

After working through this notebook, you will be able to:

- build a report from a verified metric table so that every number, sign, unit, and date is traceable to a source calculation;
- compute draft-to-source differences and isolate the sentences that fail reconciliation;
- explain why a small profit/loss mismatch is a control failure rather than a rounding detail;
- describe the draft, verified, and distributed statuses and why only an authorized user may move a report to distributed; and
- keep caveats visible when reporting scenario results such as stress losses.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script reads `data/ch32_report_validation.csv`, a frozen reconciliation table with five report metrics: portfolio market value, duration, DV01, weekly profit/loss, and stress loss, each stored with a verified source value, the value that appeared in the draft, and a unit. The script computes the draft minus source difference for every metric and isolates the failed checks. The weekly profit/loss row is the teaching case: the source says -3.2 million USD, the draft says -3.0 million USD, and the difference of 0.2 million USD must be resolved before distribution.

In [ ]:
SCRIPT = ROOT / "code" / "ch32_automated_reporting_communication.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Check the output before you trust it:

- Confirm each unit: market value, profit/loss, and stress loss in USD million; duration in years; DV01 in USD thousand per basis point.
- Verify the sign convention: negative profit/loss and negative stress loss denote losses, so a draft that writes -3.0 for a source value of -3.2 understates the loss by 0.2 million USD.
- Distinguish a genuine mismatch from a documented rounding rule; either revise the sentence or state the convention explicitly.
- Keep scenario language proportional: a stress loss of -8.6 million USD is a scenario result, not a forecast, and the sentence must say so.
- Confirm the report date, portfolio snapshot, and recipient list at the distribution gate, not only the numbers.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch32_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Require every reconciliation check to pass before verification; distribution also requires separate human approval.

In [ ]:
checks = pd.read_csv(DATA / "ch32_report_validation.csv")
verified = not checks.empty and checks.status.eq("pass").all()
approved = False  # human approval is separate from reconciliation
status = pd.DataFrame({
    "stage": ["draft", "verified", "distributed"],
    "allowed": [True, verified, verified and approved],
    "reason": ["metrics loaded", "requires all checks to pass",
               "requires checks and human approval"],
})
status

## Mini-lab: Assemble a report table from computed metrics

This lab isolates the first layer of the chapter: a report table assembled entirely from computed metrics. The illustrative portfolio has a market value of 248 million USD and a modified duration of 6.4 years. DV01 is the value change for a one basis point yield move; the stress loss applies a 50 basis point parallel spread shock; the weekly profit and loss is a realized figure of -3.2 million USD. A draft layer copies the verified metrics but rewrites the profit/loss as -3.0 million USD, and reconciliation flags the difference.

In [ ]:
import pandas as pd

value = 248_000_000  # market value in USD
duration = 6.4  # modified duration in years
shock = 0.0050  # spread shock, 50 basis points
pnl = -3_200_000  # weekly P/L in USD

metrics = pd.Series({  # computed, unit-labeled metrics
    "market_value_musd": value / 1e6,
    "dv01_kusd": value * duration * 1e-4 / 1e3,
    "stress_loss_musd": -value * duration * shock / 1e6,
    "weekly_pnl_musd": pnl / 1e6,
})
draft = metrics.copy()
draft["weekly_pnl_musd"] = -3.0  # understates the loss
report = pd.DataFrame({"source": metrics, "draft": draft})
mismatch = report.source != report.draft
report["status"] = ["fail" if m else "pass" for m in mismatch]
report

### What to observe

The computed table shows a market value of 248.0 million USD, a DV01 of 158.72 thousand USD per basis point, and a stress loss of -7.94 million USD from the 50 basis point shock. The profit/loss row fails: the source value is -3.2 million USD and the draft says -3.0, so the report is blocked at the verification gate.

## Exercises

1. **Reconcile a full draft.** Add a draft duration of 6.5 years to the table and confirm that reconciliation catches it while all other rows pass. *Hint:* Extend the draft Series and rerun the status comparison.
2. **Label every unit.** Attach an explicit unit column to the report and state why DV01 is quoted here in USD thousand per basis point. *Hint:* Keep the units in a separate Series aligned to the metric index.
3. **Document a rounding rule.** Decide whether the profit/loss draft could pass under a stated rounding convention, and write the convention as one sentence a reviewer can check. *Hint:* A rule must fix the precision and the direction of rounding.
4. **Rewrite the stress sentence.** Replace the claim that the portfolio will lose 8.6 million USD with a sentence that identifies the loss as a scenario result. *Hint:* Name the shock, the horizon, and the model assumption.
5. **Build the distribution gate.** Add a boolean that allows distribution only when every check passes and a human approval flag is set. *Hint:* Keep approval separate from reconciliation so neither can substitute for the other.

For more detail, see Chapter 32 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">